In [1]:
import test_pipeline
import slice_util
import os
import numpy as np
import glob
import pandas as pd
import shutil

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
config_dir = '/home/jhahn/puzzlefusion-plusplus/config'
files_root =  '/home/jhahn/puzzlefusion-plusplus/web/files'
ckpt_path= '/home/jhahn/puzzlefusion-plusplus/brain_lightsheet/denoiser/everyday_2000epoch/training/last.ckpt'

In [7]:
import importlib
import puzzlefusion_plusplus.denoiser.dataset.dataset
importlib.reload(puzzlefusion_plusplus.denoiser.dataset.dataset)
from puzzlefusion_plusplus.denoiser.dataset.dataset import build_test_dataloader
import test_pipeline
importlib.reload(test_pipeline)

cfg = test_pipeline.load_cfg(config_dir)

num_of_missing_slices = 0
from_index = 100
tickness = 0.002

data_ids = [f'sliced_on_1_0_0_{tickness:.3f}_True_{num_of_missing_slices}_{from_index}_700']#['0408']


tiff_dir_root, obj_dir_root, pc_dir_root, inference_dir_root, render_output_dir = test_pipeline.init_dir(files_root,data_ids)
print('tiff_dir_root',tiff_dir_root)
print('obj_dir_root',obj_dir_root)
print('pc_dir_root',pc_dir_root)
print('inference_dir_root',inference_dir_root)
print('render_output_dir',render_output_dir)

tiff_dir = f'/data/jhahn/data/brain_lightsheet/slices/{"_".join(data_ids[0].split("_")[:5])}'
tiff_list = os.listdir(tiff_dir)
tiff_list.sort(key = lambda x: int(x.split(".")[0]))
tiff_list = tiff_list[from_index:]

for _i, f in enumerate(tiff_list):
    _id = int(f.split(".")[-2])
    if _i % (num_of_missing_slices+1) == 0:
        shutil.copyfile(tiff_dir + "/" + f, tiff_dir_root + "/"+ f )    
    if _i > 10:
        break
'''
glb_dir = f'/data/jhahn/data/shape_dataset/data/brain_lightsheet/{data_ids[0]}/fractured_0'
for f in os.listdir(glb_dir):
    _id = int(f.split(".")[-2])

    shutil.copyfile(f'{tiff_dir}/{_id}.tif', f'{tiff_dir_root}/{_id}.tif')    
'''




tiff_dir_root /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/tiff
obj_dir_root /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/objs
pc_dir_root /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/pc
inference_dir_root /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/inference
render_output_dir /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render


'\nglb_dir = f\'/data/jhahn/data/shape_dataset/data/brain_lightsheet/{data_ids[0]}/fractured_0\'\nfor f in os.listdir(glb_dir):\n    _id = int(f.split(".")[-2])\n\n    shutil.copyfile(f\'{tiff_dir}/{_id}.tif\', f\'{tiff_dir_root}/{_id}.tif\')    \n'

In [8]:

import trimesh

no_gap_between_slices = True
obj_dir_list_relative = test_pipeline.tiff_2_obj(cfg, tiff_dir_root, tickness, obj_dir_root, pc_dir_root, num_of_missing_slices, no_gap_between_slices)
#obj_dir_list_relative = ['0.001_0.007']
#obj_dir_list_relative

obj_files = []
for _o in os.listdir(obj_dir_root + "/"+obj_dir_list_relative[0]):
    if not _o.endswith(".glb"):
        continue
    _glb = trimesh.load(obj_dir_root + "/"+obj_dir_list_relative[0]+"/"+_o)
    _new_obj_filename = render_output_dir+"/objs/"+_o.replace(".glb",".obj")
    _glb.export(_new_obj_filename)
    #print(_new_obj_filename)
    obj_files.append(_new_obj_filename)
slice_util.combine_obj_files(obj_files, render_output_dir+"/original.obj")

_tiff_2_obj: the number of jobs:12


tiff_2_pcd: 100%|██████████| 12/12 [00:00<00:00, 12992.17it/s]


/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/objs/test.txt
-------------------------------------
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/objs/test
frac ['fractured_0']
save_path /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/pc


Processing test data:   0%|          | 0/1 [00:00<?, ?it/s]

/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/objs/test/fractured_0 ['100.glb', '101.glb', '102.glb', '103.glb', '104.glb', '105.glb', '106.glb', '107.glb', '108.glb', '109.glb', '110.glb', '111.glb']


Processing test data: 100%|██████████| 1/1 [00:04<00:00,  4.09s/it]

test/fractured_0


[]

In [9]:
import test_pipeline
importlib.reload(test_pipeline)
test_pipeline.inference(cfg, pc_dir_root, obj_dir_list_relative, ckpt_path, inference_dir_root)
import importlib
import render_inference_result
importlib.reload(slice_util)
importlib.reload(render_inference_result)
import puzzlefusion_plusplus.denoiser.dataset.dataset
importlib.reload(puzzlefusion_plusplus.denoiser.dataset.dataset)
from puzzlefusion_plusplus.denoiser.dataset.dataset import build_test_dataloader


vertices_gt = render_inference_result.get_vertices(inference_dir_root, obj_dir_root, device=test_pipeline.device)


{'hydra': {'output_subdir': None, 'run': {'dir': '.'}}, 'defaults': ['_self_', 'denoiser/data', 'denoiser/model', 'denoiser/encoder', 'verifier/model', 'ae/model', 'ae/vq_vae', {'override hydra/hydra_logging': 'disabled'}, {'override hydra/job_logging': 'disabled'}], 'denoiser': {'ckpt_path': '/home/jhahn/puzzlefusion-plusplus/brain_lightsheet/denoiser/everyday_2000epoch/training/last.ckpt', 'data': {'val_batch_size': 1, 'matching_data_path': './data/matching_data/', 'batch_size': 64, 'num_workers': 64, 'data_fn': 'brain_lightsheet.{}.txt', 'data_dir': '/data/jhahn/data/shape_dataset/pc_data/brain_lightsheet/train/', 'data_val_dir': '/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/pc', 'mesh_data_dir': '/data/jhahn/data/shape_dataset/data', 'rot_range': -1, 'overfit': -1, 'min_num_part': 2, 'max_num_part': 20}, 'ae': {'ae_name': {'_target_': 'puzzlefusion_plusplus.denoiser.model.modules.encoder.VQVAE'}, 'n_embeddings': 1024, 'embedding_dim': 16, 'num_po

100%|██████████| 1/1 [00:00<00:00, 585.39it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-e5a11831-23aa-537c-9ebe-5e6cce8a2bce,MIG-6b03ab11-3f04-52a2-952d-e8d9df38ee72]


Testing: |          | 0/? [00:00<?, ?it/s]

_save_inference_data /home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/inference/0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      eval/part_acc                 0.0
       eval/rmse_r          25.571990966796875
       eval/rmse_t          0.00532158138230443
      eval/shape_cd         0.01402987539768219
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


100%|██████████| 12/12 [00:00<00:00, 565.05it/s]


In [10]:
#test_pipeline.inference(cfg, pc_dir_root, obj_dir_list_relative, ckpt_path, inference_dir_root)
import importlib
import render_inference_result
importlib.reload(slice_util)
importlib.reload(render_inference_result)
import puzzlefusion_plusplus.denoiser.dataset.dataset
importlib.reload(puzzlefusion_plusplus.denoiser.dataset.dataset)
from puzzlefusion_plusplus.denoiser.dataset.dataset import build_test_dataloader
import test_pipeline
importlib.reload(test_pipeline)

test_pipeline.render(inference_dir_root, vertices_gt, render_output_dir)
shape_cd = test_pipeline.eval( vertices_gt,inference_dir_root,render_output_dir)
shape_cd

100%|██████████| 12/12 [00:01<00:00, 11.43it/s]


/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/0.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/1.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/2.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/3.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/4.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/5.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/6.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/7.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/render/0/trans/8.obj
/home/jhahn/puzzlefusion-plusplus/web/files/sliced_on_1_0_0_0.002_True_0_100_700/r

100%|██████████| 12/12 [00:00<00:00, 35.07it/s]


tensor([0.0023, 0.0026, 0.0027, 0.0010, 0.0030, 0.0030, 0.0013, 0.0027, 0.0015,
        0.0011, 0.0026, 0.0013], device='cuda:0')